# RE_102 · Quina música fa ballar

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 1**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes en una startup que fa playlists per a gimnasos, botigues i festes. L'equip de
producte vol entendre el catàleg abans de fer un recomanador, i et passa unes 30.000 cançons de
Spotify amb les seves característiques d'àudio.

**La pregunta:** quin gènere fa més ballar i quin té més energia per entrenar? I, donada una cançó
que t'agrada, quines cinc del catàleg s'hi assemblen més?

L'enunciat complet, amb el diccionari de columnes, és a [RE_102](../RE_102_spotify.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
df = pd.read_csv(URL)
print(df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma i tipus

Quantes files i columnes té? Quines columnes són de text i quines numèriques? Quants gèneres i quants subgèneres de playlist hi ha?

In [ ]:
print(df.shape)
print(df.dtypes.value_counts())
print(df["playlist_genre"].nunique(), df["playlist_subgenre"].nunique())
print(df["playlist_genre"].value_counts())

### 1.2 · Nuls

Quantes cel·les buides hi ha a cada columna? Mostra les files que en tenen. Tenen alguna cosa en comú?

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0])
df[df["track_name"].isna()]

### 1.3 · Cançons repetides

Cada cançó té un identificador, `track_id`. Quantes files hi ha i quants `track_id` diferents? Quantes files repeteixen una cançó que ja ha sortit abans? Busca una cançó que surti diverses vegades i mira en què es diferencien les files.

In [ ]:
print(len(df), df["track_id"].nunique(), df.duplicated(subset=["track_id"]).sum())
exemple = df.loc[df.duplicated("track_id"), "track_id"].iloc[0]
df[df["track_id"] == exemple][["track_name", "track_artist", "playlist_name", "playlist_genre", "playlist_subgenre"]]

**Comentari.** Hi ha 4.477 files que repeteixen un `track_id`: la mateixa cançó surt a diverses playlists, sovint de gèneres diferents. A més, la mateixa cançó pot tenir `track_id` diferents (single, àlbum, recopilatori).

## Nivell 2 · Respondre

### 2.1 · El gènere més ballable

Calcula la `danceability` mitjana de cada gènere, ordena de més a menys i fes-ne un gràfic de barres.

In [ ]:
ball = df.groupby("playlist_genre")["danceability"].mean().sort_values(ascending=False)
print(ball.round(3))
ball.plot(kind="bar", title="Danceability mitjana per gènere")
plt.show()

### 2.2 · Per entrenar

Per entrenar es busca energia i ritme. En una sola taula, calcula per gènere la mitjana i la mediana d'`energy` i de `tempo`. Quin gènere triaries per a un gimnàs? Té en compte les dues columnes?

In [ ]:
entrenar = df.groupby("playlist_genre")[["energy", "tempo"]].agg(["mean", "median"]).round(3)
print(entrenar.shape)
entrenar.sort_values(("energy", "mean"), ascending=False)

### 2.3 · Sense repetides

Repeteix 2.1 amb una taula on cada `track_id` surti **una sola vegada**. Canvia l'ordre dels gèneres? Canvien gaire les mitjanes? Escriu en una línia quina de les dues versions faries servir i per què.

In [ ]:
unics = df.drop_duplicates(subset=["track_id"])
print(len(unics), df["track_id"].nunique())
comparacio = pd.DataFrame({
    "totes": df.groupby("playlist_genre")["danceability"].mean(),
    "unics": unics.groupby("playlist_genre")["danceability"].mean(),
}).round(3)
comparacio.sort_values("totes", ascending=False)

**Comentari.** L'ordre no canvia i les mitjanes es mouen poc. El que canvia és el gènere que "es queda" la cançó: `drop_duplicates` es queda amb la primera aparició, que depèn de l'ordre del fitxer.

### 2.4 · Cançons cada cop més curtes?

La data del disc és a `track_album_release_date`. Crea una columna `any` (enter) i calcula la durada mitjana **en minuts** per any. Fes-ne un gràfic de línia. Amb pocs discos, un any surt molt soroll: dibuixa només els anys amb almenys 50 cançons.

In [ ]:
print(df["track_album_release_date"].str.len().value_counts())
df["any"] = df["track_album_release_date"].str[:4].astype(int)
df["minuts"] = df["duration_ms"] / 60000
print(df["any"].isna().sum(), df["any"].min(), df["any"].max(), len(df))
per_any = df.groupby("any")["minuts"].agg(["mean", "size"])
per_any = per_any[per_any["size"] >= 50]
print(per_any.round(2).tail(10))
per_any["mean"].plot(kind="line", title="Durada mitjana (min) per any del disc")
plt.ylabel("minuts")
plt.show()

**Comentari.** Les dates barregen tres formats (`AAAA-MM-DD`, `AAAA` i `AAAA-MM`), però els quatre primers caràcters sempre són l'any. La durada baixa clarament als últims anys.

### 2.5 · Un recomanador amb NumPy

Fes servir aquestes nou columnes d'àudio:

`["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]`

1. Passa-les a un array `X` (fes-ho sobre la taula sense repetides de 2.3).
2. `loudness` i `tempo` tenen escales molt diferents de la resta. **Normalitza** cada columna al
   rang [0, 1] amb el mínim i el màxim de cada columna (`axis=0`).
3. Tria una cançó que t'agradi (busca-la pel títol) i calcula la distància euclidiana de la seva fila
   a totes les altres.
4. Mostra les cinc cançons més properes: títol, artista i gènere.

In [ ]:
audio = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
unics = unics.reset_index(drop=True)
X = unics[audio].to_numpy()
Xn = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print(X.shape, Xn.min(axis=0), Xn.max(axis=0))

trobades = unics[unics["track_name"].str.contains("Blinding Lights", case=False, na=False)]
print(trobades[["track_name", "track_artist"]])
i = trobades.index[0]
d = np.linalg.norm(Xn - Xn[i], axis=1)
print("distància a si mateixa:", d[i])
properes = np.argsort(d)[1:6]
unics.loc[properes, ["track_name", "track_artist", "playlist_genre"]].assign(distancia=d[properes].round(3))

**Comentari.** Si no es treuen les repetides, les primeres "recomanacions" són la mateixa cançó a distància 0. Si no es normalitza, `tempo` i `loudness` dominen la distància i el recomanador busca només cançons amb el mateix tempo.

## Nivell 3 · Més enllà

### 3.1 · Ballable vol dir popular?

Parteix `danceability` en cinc trams i calcula la popularitat mitjana de cada tram. Les cançons més ballables són més populars? Fes-ho amb la taula sense repetides.

In [ ]:
trams = unics.groupby(pd.cut(unics["danceability"], 5), observed=True)["track_popularity"].agg(["mean", "size"])
print(trams.round(1), trams["size"].sum(), len(unics))

### 3.2 · El subgènere més trist

`valence` mesura com de "positiva" sona una cançó. Quins són els tres subgèneres amb la `valence` mitjana més baixa i els tres amb la més alta?

In [ ]:
val = df.groupby("playlist_subgenre")["valence"].mean().sort_values()
print(val.head(3).round(3))
print(val.tail(3).round(3))

## Conclusió de referència

El rap (0,718) i el llatí (0,713) són els gèneres més ballables de mitjana, i el rock el que menys
(0,521). Per entrenar, l'EDM té l'energia més alta (mitjana 0,80) i el tempo més alt (126 bpm). Les
32.833 files són 28.356 cançons diferents: 4.477 files repeteixen un `track_id` perquè la cançó surt
a diverses playlists, però treure-les no canvia l'ordre dels gèneres. La durada mitjana ha baixat
d'uns 4,2 minuts (discos del 2011) a 3,3 (2019-2020). Amb les nou característiques normalitzades,
la cançó més propera a *Blinding Lights* és *Part-Time Lover* (Dabin), a 0,058.